In [1]:
import pandas as pd
import glob
import os

In [2]:
# =========================================================
# 2. 연도별 데이터 불러오기
# =========================================================
dfs = {}

files = glob.glob("data/*.csv")

for file in files:

    # 파일명에서 연도 추출
    year = os.path.basename(file).split("_")[0]

    # csv 읽기
    df = pd.read_csv(file)

    # dictionary 저장
    dfs[year] = df

    print(year, df.shape)

2014 (11016, 165)
2015 (21792, 165)
2016 (50568, 165)
2017 (68424, 165)
2018 (80844, 165)
2019 (97500, 165)
2020 (117012, 165)
2021 (148464, 165)


C:\Users\psubi\AppData\Local\Temp\ipykernel_36596\3290997338.py:14: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


2022 (148980, 165)
2023 (133428, 165)
2024 (122388, 165)
2025 (120636, 165)
2026 (50616, 165)


In [3]:
# =========================================================
# 3. 각 연도별 컬럼 개수 확인
# =========================================================

print("\n========== 연도별 컬럼 개수 ==========\n")

for year, df in dfs.items():
    print(f"{year}: {len(df.columns)}개")


========== 연도별 컬럼 개수 ==========

2014: 165개
2015: 165개
2016: 165개
2017: 165개
2018: 165개
2019: 165개
2020: 165개
2021: 165개
2022: 165개
2023: 165개
2024: 165개
2025: 165개
2026: 165개


In [4]:
# =========================================================
# 4. 모든 연도에 공통으로 존재하는 컬럼 찾기 -> 확인해보니 모든 컬럼들이 다 같음 -> 그냥 concat해서 진행해도 될듯
# =========================================================

common_columns = set(list(dfs.values())[0].columns)

for df in list(dfs.values())[1:]:
    common_columns = common_columns.intersection(df.columns)

common_columns = sorted(list(common_columns))

print("\n========== 공통 컬럼 개수 ==========\n")
print(len(common_columns))


print("\n========== 공통 컬럼 목록 ==========\n")

for col in common_columns:
    print(col)


========== 공통 컬럼 개수 ==========

165

========== 공통 컬럼 목록 ==========

assists
assistsat10
assistsat15
assistsat20
assistsat25
atakhans
ban1
ban2
ban3
ban4
ban5
barons
champion
chemtechs
ckpm
clouds
controlwardsbought
csat10
csat15
csat20
csat25
csdiffat10
csdiffat15
csdiffat20
csdiffat25
cspm
damagemitigatedperminute
damageshare
damagetakenperminute
damagetochampions
damagetotowers
datacompleteness
date
deaths
deathsat10
deathsat15
deathsat20
deathsat25
doublekills
dpm
dragons
dragons (type unknown)
earned gpm
earnedgold
earnedgoldshare
elders
elementaldrakes
firstPick
firstbaron
firstblood
firstbloodassist
firstbloodkill
firstbloodvictim
firstdragon
firstherald
firstmidtower
firsttothreetowers
firsttower
game
gameid
gamelength
goldat10
goldat15
goldat20
goldat25
golddiffat10
golddiffat15
golddiffat20
golddiffat25
goldspent
gpr
gspd
heralds
hextechs
infernals
inhibitors
kills
killsat10
killsat15
killsat20
killsat25
league
minionkills
monsterkills
monsterkillsenemyjungle
monsterkillsown

In [5]:
# =========================================================
# 5. 연도별로만 존재하는 컬럼 확인
# =========================================================

print("\n========== 연도별 특수 컬럼 ==========\n")

for year, df in dfs.items():

    unique_cols = set(df.columns) - set(common_columns)

    print(f"\n[{year} 전용 컬럼]")

    for col in sorted(unique_cols):
        print(col)



========== 연도별 특수 컬럼 ==========


[2014 전용 컬럼]

[2015 전용 컬럼]

[2016 전용 컬럼]

[2017 전용 컬럼]

[2018 전용 컬럼]

[2019 전용 컬럼]

[2020 전용 컬럼]

[2021 전용 컬럼]

[2022 전용 컬럼]

[2023 전용 컬럼]

[2024 전용 컬럼]

[2025 전용 컬럼]

[2026 전용 컬럼]


In [6]:
selected_features = [

    # =========================================================
    # 1. 기본 정보 컬럼
    # =========================================================

    "playername",     # Player : 선수 이름 -> 선수별 집계 및 분석에 필요
    "teamname",       # Team : 팀 이름 -> 팀 정보 확인
    "position",       # Pos : 포지션 -> 포지션별 정규화 수행 예정

    # "champion",     # Champion : 챔피언 이름 -> 메타 영향이 너무 커 제외
    # "event",        # Event : 대회명 -> 현재 LCK만 사용할 예정이라 제외
    # "rank",         # Rank : 솔로랭크 랭킹 -> 프로 경기 분석과 직접 연결 어려움
    # "lp",           # LP : 래더 포인트 -> 솔랭 데이터 성격이라 제외

    "patch",          # 패치 버전 -> 패치별 메타 차이 보정을 위해 사용
    "year",           # 연도 -> 연도별 메타 차이 보정을 위해 사용
    "date",           # 경기 날짜 -> 시간 흐름에 따른 트렌드 분석 가능


    # =========================================================
    # 2. 라인전 체급 관련 컬럼
    # =========================================================

    "golddiffat15",   # GD15 : 15분 골드 차이 -> 라인전 우위 판단 핵심 지표
    "xpdiffat15",     # XPD15 : 15분 경험치 차이 -> 성장 우위 반영
    "csdiffat15",     # CSD15 : 15분 CS 차이 -> 라인전 압박 능력 판단

    # "golddiffat10", # GD10 : 초반 골드 차이 -> GD15와 정보 중복 가능성
    # "golddiffat20", # GD20 : 중후반 영향 포함 -> 라인전 체급 의미 약화

    # "xpdiffat10",   # XPD10 : XPD15와 정보 중복 가능성
    # "xpdiffat20",   # XPD20 : 중후반 영향 포함

    # "csdiffat10",   # CSD10 : CSD15와 정보 중복 가능성
    # "csdiffat20",   # CSD20 : 중후반 영향 포함

    # "gxdiffat15",   # GXD15 : 골드+경험치 통합 지표 -> GD/XPD와 정보 중복 가능성 큼

    "goldat15",       # 15분 골드량 -> 성장 속도 반영
    "xpat15",         # 15분 경험치량 -> 성장량 반영
    "csat15",         # 15분 CS량 -> 안정적 수급 능력 반영

    # "cs%p15",       # CS%P15 : 팀 CS 비중 -> 팀 스타일 영향 큼
    # "lne%",         # LNE% : Lane Control -> 해석 모호 및 중복 가능성


    # =========================================================
    # 3. 전투력 / 캐리력 관련 컬럼
    # =========================================================

    "kills",          # K : 킬 수 -> 캐리력 판단
    "deaths",         # D : 데스 수 -> 안정성 판단
    "assists",        # A : 어시스트 수 -> 교전 기여도 반영

    "teamkills",      # 팀 전체 킬 수 -> KP 계산용으로 사용 예정
    
    # "kd",           # KD : 킬데스 비율 -> KDA와 정보 중복
    # "kpg",          # 경기당 킬 -> 경기 수 영향 큼
    # "apg",          # 경기당 어시스트 -> 경기 수 영향 큼
    # "dpg",          # 경기당 데스 -> 경기 수 영향 큼

    # "ks%",          # 킬 비중 -> 원딜 포지션 편향 심함

    "dpm",            # DPM : 분당 챔피언 피해량 -> 핵심 캐리력 지표
    "damageshare",    # DMG% : 팀 내 딜 비중 -> 팀 내 영향력 판단

    "damagetakenperminute",      # Damage Taken per minute -> 전면 교전 참여도
    "damagemitigatedperminute",  # Damage Mitigated per minute -> 탱커 가치 반영

    # "dmg%p15",      # 15분 이후 딜 비중 -> 후반 메타 영향 큼
    # "d%p15",        # 후반 딜 비중 -> 중복 가능성

    # "ckpm",         # 경기 전체 교전 빈도 -> 개인 역량보다 경기 스타일 영향 큼


    # =========================================================
    # 4. 시야 및 운영 기여 컬럼
    # =========================================================

    "visionscore",    # Vision Score -> 맵 장악 및 운영 능력 판단
    "vspm",           # VSPM : 분당 Vision Score -> 시야 효율 판단

    "wardsplaced",    # 설치 와드 수 -> 시야 확보 능력
    "wardskilled",    # 제거 와드 수 -> 상대 시야 제거 능력

    "wpm",            # WPM : 분당 와드 설치 수 -> 시야 장악 효율
    "wcpm",           # WCPM : 분당 와드 제거 수 -> 시야 제거 효율

    "controlwardsbought", # CWPM 관련 원본 값 -> 운영 기여 반영

    # "wc%",          # 와드 제거율 -> 퍼센트 기반이라 해석 어려움
    # "vwc%",         # visible ward 제거율 -> 세부 의미 약함
    # "iwc%",         # invisible ward 제거율 -> 데이터 안정성 낮음


    # =========================================================
    # 5. 자원 수급 및 성장 효율 컬럼
    # =========================================================

    "earnedgold",         # 획득 골드 -> 실제 성장량 반영
    "earned gpm",         # EGPM : 분당 획득 골드 -> 성장 효율 핵심

    "earnedgoldshare",    # GOLD% : 팀 내 골드 비중 -> 자원 집중도 반영

    # "gpm",              # 분당 골드 -> EGPM과 정보 중복 가능성

    # "gpr",                # GPR : 골드 점유율 -> 자원 우위 판단 -> 11 단계에서 확인 결과 년도별로 존재하지 않는 경우가 있어 제외
    # "gspd",               # GSPD : 골드 소비 차이 -> 자원 활용 효율 -> 11 단계에서 확인 결과 년도별로 존재하지 않는 경우가 있어 제외

    # "total cs",         # 총 CS -> 경기 시간 영향 너무 큼
    "cspm",               # CSPM : 분당 CS -> 안정적 성장 능력

    # "jng%",             # 정글 점유율 -> 정글 포지션 편향 심함


    # =========================================================
    # 6. 탱킹 / 생존 관련 컬럼
    # =========================================================

    # damagetakenperminute 사용
    # damagemitigatedperminute 사용
    # -> 탱커 가치 및 전면 교전 참여도 반영


    # =========================================================
    # 7. 오브젝트 관련 컬럼
    # =========================================================

    # "bn%",             # 바론 점유율 -> 팀 오더 영향 큼
    # "drg%",            # 드래곤 점유율 -> 팀 운영 영향 큼
    # "hld%",            # 전령 점유율 -> 팀 macro 영향 큼
    # "eld%",            # 장로 점유율 -> 경기 상황 영향 큼
    # "grb%",            # 공허 유충 점유율 -> 최신 메타 편향

    # "stl",             # 오브젝트 스틸 -> 표본 수 너무 적음
    # "stlpg",           # 경기당 오브젝트 스틸 -> 표본 수 적음

    # "ft%",             # 첫 타워 비율 -> 팀 전략 영향
    # "fd%",             # 첫 드래곤 비율 -> 팀 전략 영향
    # "fbn%",            # 첫 바론 비율 -> 팀 전략 영향
    # "f3t%",            # 첫 3타워 비율 -> 팀 운영 영향 큼

    # "turretplates",      # PPG : 포탑 방패 획득 -> 초반 압박력 반영 -> 11 단계에서 확인 결과 년도별로 존재하지 않는 경우가 있어 제외
    "damagetotowers",    # TDPG : 타워 피해량 -> 운영 기여도 반영


    # =========================================================
    # 8. 초반 영향력 관련 컬럼
    # =========================================================

    "firstbloodkill",    # First Blood 킬 여부 -> 초반 aggressive 성향
    "firstbloodassist",  # First Blood 어시스트 여부 -> 초반 교전 기여
    "firstbloodvictim",  # First Blood 희생 여부 -> 초반 안정성 판단


    # =========================================================
    # 9. Oracle’s Elixir 자체 평가 지표
    # =========================================================

    "league",            # 현재는 LCK만 필터링에 사용
    "datacompleteness",  # complete 경기만 사용하기 위해 필요
    "gamelength",        # remake 제거를 위한 경기 시간 정보
    "result"             # 승패 분석 및 추가 실험 가능성 고려

    # =========================================================
    # 10. 필터링용 컬럼(다 제외)
    # =========================================================
    # "OE Rating",
    # "OE Grade",
    # "EGR",
    # "MLG"
]

In [7]:
# =========================================================
# 6. 연도별로만 선택 컬럼만 남기기
# =========================================================
for year in dfs:

    dfs[year] = dfs[year][selected_features]

    print(year, dfs[year].shape)

2014 (11016, 39)
2015 (21792, 39)
2016 (50568, 39)
2017 (68424, 39)
2018 (80844, 39)
2019 (97500, 39)
2020 (117012, 39)
2021 (148464, 39)
2022 (148980, 39)
2023 (133428, 39)
2024 (122388, 39)
2025 (120636, 39)
2026 (50616, 39)


In [8]:
# =========================================================
# 7. lck 경기만 남기기
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["league"] == "LCK"
    ]
    print(year, dfs[year].shape)

2014 (0, 39)
2015 (96, 39)
2016 (5880, 39)
2017 (6120, 39)
2018 (6096, 39)
2019 (5952, 39)
2020 (5796, 39)
2021 (5868, 39)
2022 (5604, 39)
2023 (5856, 39)
2024 (5784, 39)
2025 (6660, 39)
2026 (2904, 39)


In [9]:
# =========================================================
# 8. 불완전한 경기 제거 (값에 decomplete이 들어간 경기 제거)
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["datacompleteness"] == "complete"
    ]
    print(year, dfs[year].shape)

2014 (0, 39)
2015 (96, 39)
2016 (5880, 39)
2017 (6120, 39)
2018 (6096, 39)
2019 (5940, 39)
2020 (5748, 39)
2021 (5868, 39)
2022 (5604, 39)
2023 (5856, 39)
2024 (5784, 39)
2025 (6660, 39)
2026 (2904, 39)


In [10]:
# =========================================================
# 9. 15분 미만의 비정상 경기 제거 -> 15분 미만의 경기는 경기 도중에 문제가 생겨서 중단된 경기들임. -> 확인 결과 15분 미만의 경기가 없긴 했음. 이상치 없는거 확인. 그리고 lck 내부에서는 정상 경기라면 일단 15분 내의 경기가 불가능.
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["gamelength"] >= 900
    ]
    print(year, dfs[year].shape)

2014 (0, 39)
2015 (96, 39)
2016 (5880, 39)
2017 (6120, 39)
2018 (6096, 39)
2019 (5940, 39)
2020 (5748, 39)
2021 (5868, 39)
2022 (5604, 39)
2023 (5844, 39)
2024 (5784, 39)
2025 (6660, 39)
2026 (2904, 39)


In [11]:
print(
    dfs[year]["gamelength"].min()
)

1366


In [12]:
# =========================================================
# 10. 팀 단위 row 제거 -> 선수 개인의 기량만 판단하기 때문에 필요 없다고 판단. -> 각 라인별로의 경기만 남게 됨.
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["position"] != "team"
    ]
    print(year, dfs[year].shape)

2014 (0, 39)
2015 (80, 39)
2016 (4900, 39)
2017 (5100, 39)
2018 (5080, 39)
2019 (4950, 39)
2020 (4790, 39)
2021 (4890, 39)
2022 (4670, 39)
2023 (4870, 39)
2024 (4820, 39)
2025 (5550, 39)
2026 (2420, 39)


In [13]:
print(
    dfs[year]["position"].unique()
)

<StringArray>
['top', 'jng', 'mid', 'bot', 'sup']
Length: 5, dtype: str


In [14]:
# =========================================================
# 연도별 현재 데이터 개수 확인 (확인용임)
# =========================================================

for year in sorted(dfs.keys()):

    print(
        f"{year} : "
        f"{dfs[year].shape[0]} rows, "
        f"{dfs[year].shape[1]} columns"
    )

2014 : 0 rows, 39 columns
2015 : 80 rows, 39 columns
2016 : 4900 rows, 39 columns
2017 : 5100 rows, 39 columns
2018 : 5080 rows, 39 columns
2019 : 4950 rows, 39 columns
2020 : 4790 rows, 39 columns
2021 : 4890 rows, 39 columns
2022 : 4670 rows, 39 columns
2023 : 4870 rows, 39 columns
2024 : 4820 rows, 39 columns
2025 : 5550 rows, 39 columns
2026 : 2420 rows, 39 columns


In [15]:
# =========================================================
# 11 .연도별 결측치 확인
# =========================================================
for year in dfs:

    print(f"\n===== {year} =====")

    missing = dfs[year].isnull().sum()

    # 결측치 있는 컬럼만 출력
    missing = missing[missing > 0]

    print(missing)


===== 2014 =====
Series([], dtype: int64)

===== 2015 =====
Series([], dtype: int64)

===== 2016 =====
Series([], dtype: int64)

===== 2017 =====
Series([], dtype: int64)

===== 2018 =====
Series([], dtype: int64)

===== 2019 =====
Series([], dtype: int64)

===== 2020 =====
Series([], dtype: int64)

===== 2021 =====
playername    5
dtype: int64

===== 2022 =====
Series([], dtype: int64)

===== 2023 =====
Series([], dtype: int64)

===== 2024 =====
Series([], dtype: int64)

===== 2025 =====
Series([], dtype: int64)

===== 2026 =====
Series([], dtype: int64)


In [16]:
# =========================================================
# 모든 시즌에 공통으로 존재하는 컬럼 찾기
# =========================================================
common_columns = None

for year in dfs:

    cols = set(dfs[year].columns)

    if common_columns is None:
        common_columns = cols

    else:
        common_columns = (
            common_columns & cols
        )

print(sorted(common_columns))

['assists', 'controlwardsbought', 'csat15', 'csdiffat15', 'cspm', 'damagemitigatedperminute', 'damageshare', 'damagetakenperminute', 'damagetotowers', 'datacompleteness', 'date', 'deaths', 'dpm', 'earned gpm', 'earnedgold', 'earnedgoldshare', 'firstbloodassist', 'firstbloodkill', 'firstbloodvictim', 'gamelength', 'goldat15', 'golddiffat15', 'kills', 'league', 'patch', 'playername', 'position', 'result', 'teamkills', 'teamname', 'visionscore', 'vspm', 'wardskilled', 'wardsplaced', 'wcpm', 'wpm', 'xpat15', 'xpdiffat15', 'year']


In [17]:
# =========================================================
# selected_features 중 없는 컬럼 확인
# =========================================================

missing_features = []

for col in selected_features:

    if col not in common_columns:

        missing_features.append(col)

print(missing_features)

[]


In [18]:
# =========================================================
# 연도별 결측치 비율 확인 -> 결측치가 있는 컬럼을 확인했는데 gpr, gspd, turretplates가 100% 였음. 즉 다른 년도에는 존재하지 않는 데이터임 -> selected features에서 제거!
# =========================================================

for year in dfs:

    print(f"\n===== {year} =====")

    missing_ratio = (
        dfs[year].isnull().mean() * 100
    )

    missing_ratio = (
        missing_ratio[missing_ratio > 0]
    )

    print(
        missing_ratio.sort_values(
            ascending=False
        )
    )


===== 2014 =====
Series([], dtype: float64)

===== 2015 =====
Series([], dtype: float64)

===== 2016 =====
Series([], dtype: float64)

===== 2017 =====
Series([], dtype: float64)

===== 2018 =====
Series([], dtype: float64)

===== 2019 =====
Series([], dtype: float64)

===== 2020 =====
Series([], dtype: float64)

===== 2021 =====
playername    0.102249
dtype: float64

===== 2022 =====
Series([], dtype: float64)

===== 2023 =====
Series([], dtype: float64)

===== 2024 =====
Series([], dtype: float64)

===== 2025 =====
Series([], dtype: float64)

===== 2026 =====
Series([], dtype: float64)


In [19]:
# =========================================================
# 2021 playername 결측 row 확인
# =========================================================

missing_rows = dfs["2021"][
    dfs["2021"]["playername"].isnull()
]

print(missing_rows)

       playername teamname position  patch  year                 date  \
132663        NaN       T1      bot  11.16  2021  2021-09-02 07:52:07   
132687        NaN       T1      bot  11.16  2021  2021-09-02 08:41:36   
132723        NaN       T1      bot  11.16  2021  2021-09-02 09:35:53   
132747        NaN       T1      bot  11.16  2021  2021-09-02 10:30:09   
132788        NaN       T1      bot  11.16  2021  2021-09-02 11:26:56   

        golddiffat15  xpdiffat15  csdiffat15  goldat15  ...  earnedgoldshare  \
132663        -161.0      -135.0        -8.0    4887.0  ...         0.252907   
132687        -188.0         9.0         2.0    5244.0  ...         0.283789   
132723        1633.0       103.0        10.0    6633.0  ...         0.295009   
132747        -917.0     -1376.0       -30.0    4425.0  ...         0.241791   
132788         920.0       417.0         4.0    5630.0  ...         0.256350   

           cspm  damagetotowers  firstbloodkill  firstbloodassist  \
132663  10.

In [ ]:
# 직접 빈 5개의 값을 채워넣으려고 하는데 일단 선수가 누구인지 확인해야함. -> 찾아본 결과 해당 날짜는 구마유시였음.

missing_rows[
    [
        "date",
        "teamname",
        "result",
        "patch"
    ]
]

,date,teamname,result,patch
132663,2021-09-02 07:52:07,T1,1,11.16
132687,2021-09-02 08:41:36,T1,1,11.16
132723,2021-09-02 09:35:53,T1,0,11.16
132747,2021-09-02 10:30:09,T1,0,11.16
132788,2021-09-02 11:26:56,T1,1,11.16


In [21]:
# =========================================================
# 확인된 Gumayusi row만 수정
# =========================================================

gumayusi_idx = [
    132663,
    132687,
    132723,
    132747,
    132788
]

dfs["2021"].loc[
    gumayusi_idx,
    "playername"
] = "Gumayusi"

In [ ]:
# 잘 들어갔는지 확인하고
dfs["2021"].loc[
    gumayusi_idx,
    ["playername", "teamname", "position"]
]

,playername,teamname,position
132663,Gumayusi,T1,bot
132687,Gumayusi,T1,bot
132723,Gumayusi,T1,bot
132747,Gumayusi,T1,bot
132788,Gumayusi,T1,bot


In [ ]:
# 다시 결측 확인
dfs["2021"]["playername"].isnull().sum()

np.int64(0)

In [24]:
# =========================================================
# 저장 폴더 생성
# =========================================================

os.makedirs(
    "processed",
    exist_ok=True
)

# =========================================================
# 연도별 csv 저장
# =========================================================

for year in dfs:

    save_path = (
        f"processed/{year}_clean.csv"
    )

    dfs[year].to_csv(
        save_path,
        index=False
    )

    print(f"{save_path} 저장 완료")

processed/2014_clean.csv 저장 완료
processed/2015_clean.csv 저장 완료
processed/2016_clean.csv 저장 완료
processed/2017_clean.csv 저장 완료
processed/2018_clean.csv 저장 완료
processed/2019_clean.csv 저장 완료
processed/2020_clean.csv 저장 완료
processed/2021_clean.csv 저장 완료
processed/2022_clean.csv 저장 완료
processed/2023_clean.csv 저장 완료
processed/2024_clean.csv 저장 완료
processed/2025_clean.csv 저장 완료
processed/2026_clean.csv 저장 완료
